In [1]:
import time
import numpy as np
from pynq import Overlay, allocate

# --- Overlay + IP handles ---
ol      = Overlay("rdo_filter_bd_wrapper.bit")  # adjust path if needed
rdo     = ol.rdo_filter_axi_0
ns_flt  = ol.ns_filter_0
pc_flt  = ol.pc_filter_0
dma_ns  = ol.axi_dma_4    # per your BD
dma_pc  = ol.axi_dma_5    # per your BD

# --- Frame geometry ---
H, W = 1080, 1920

# --- Data paths ---
BASE   = '/home/xilinx/jupyter_notebooks/pc_filter'
LR_DIR = f'{BASE}/TreesAndGrass_1920_1080_30fps_8bit/TEST_PRELR/RANGE_1'
HR_DIR = f'{BASE}/TreesAndGrass_1920_1080_30fps_8bit/TEST_HR'

# --- PC coefficient model ---
data        = np.load(f'{BASE}/pc_qp_group_1.npz')
T_float     = data['T']
LUT         = data['LUT']
cluster_map = data['cluster_map']
LUT_fixed   = np.round(LUT * 8192).astype(np.int32)

# --- DMA helpers ---
def reset_dma(dma):
    mm = dma.mmio
    mm.write(0x00, 0x4); mm.write(0x30, 0x4)
    time.sleep(0.01)
    mm.write(0x00, 0x1); mm.write(0x30, 0x1)
    dma.sendchannel._first_transfer = True
    dma.recvchannel._first_transfer = True

def load_raw_y(path):
    with open(path, 'rb') as f:
        return np.frombuffer(f.read(W*H), dtype=np.uint8).reshape(H, W).copy()

def psnr(a, b):
    m = float(np.mean((a.astype(np.int32) - b.astype(np.int32))**2))
    return float('inf') if m == 0 else 10*np.log10(255**2/m)

print("Setup ready.")

Setup ready.


In [ ]:
def run_ns_v5(sr_uint8, taps, norm):
    # Load identity taps into every bank (0..39 covers all possible RU positions)
    for bank in range(40):
        ns_flt.write(0x3C, bank << 8)          # select bank
        for i in range(12):
            ns_flt.write(i*4, int(taps[i]) & 0x7F)
        ns_flt.write(0x30, int(norm) & 0xFFFF)

    ns_flt.write(0x34, W)
    ns_flt.write(0x38, H)
    ns_flt.write(0x3C, 0x2)                     # FLIP: staged→active

    # DMA
    ns_in_buf  = allocate(shape=(H*W,),     dtype=np.uint32)
    ns_out_buf = allocate(shape=((H-6)*W,), dtype=np.uint32)
    ns_in_buf[:] = sr_uint8.flatten().astype(np.uint32)

    mm = dma_ns.mmio
    mm.write(0x00, 0x4); mm.write(0x30, 0x4); time.sleep(0.01)
    mm.write(0x00, 0x1); mm.write(0x30, 0x1)
    dma_ns.sendchannel._first_transfer = True
    dma_ns.recvchannel._first_transfer = True

    dma_ns.recvchannel.transfer(ns_out_buf)
    dma_ns.sendchannel.transfer(ns_in_buf)
    ns_flt.write(0x3C, 0x1)                     # start
    dma_ns.sendchannel.wait()
    dma_ns.recvchannel.wait()

    hw = (ns_out_buf & 0xFF).astype(np.uint8).reshape(H-6, W).copy()
    ns_in_buf.freebuffer(); ns_out_buf.freebuffer()
    return hw

# Reload if needed
prelr = load_raw_y(f'{LR_DIR}/frame_013_qp160_prelr.yuv')

# Restart kernel first, redo setup, load prelr, then:
ns_out = run_ns_v5(prelr, [0]*11 + [32], 2048)

# Identity check
start = 3*W + 3
n = ns_out.size
hw_flat = ns_out.flatten()
exp_flat = prelr.flatten()[start : start + n]
diff = hw_flat.astype(np.int16) - exp_flat.astype(np.int16)
core = diff[:-20]
print(f"NS identity with ALL 40 banks + FLIP:")
print(f"  max |diff|:  {int(np.abs(core).max())}")
print(f"  mean |diff|: {float(np.abs(core).mean()):.4f}")
print(f"  exact match: {float((core == 0).mean())*100:.2f}%")
print(f"  pixel_count: {ns_flt.read(0x00)}")